# 01 · Data Understanding

**Goal:** understand exactly what is in the dataset before any modelling — in particular,
what the treatment `T`, the outcome `Y` and the covariates `X` are, and whether the data
can support a causal question.

**Dataset:** Orange Belgium churn-uplift benchmark (Verhelst et al., *A churn prediction dataset
from the telecom sector: a new benchmark for uplift modeling*, 2023, arXiv:2312.07206),
distributed via OpenML (id 45580, `churn-uplift-mlg`) under CC BY-NC-ND 4.0.
The raw data is downloaded on first run and cached in `data/raw/` (not committed).

> The dataset comes from a telecom operator, not a bank. It is used here because it is one of very
> few public datasets with a *real* randomised retention campaign. The analytical chain
> (experiment → effect → heterogeneity → targeting) transfers directly to a bank's retention problem.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.functions import load_data, feature_lists, TREATMENT, OUTCOME

pd.set_option("display.max_columns", 30)
df = load_data()

## 1. Load and inspect

In [2]:
print("Shape:", df.shape)
df.head()

Shape: (11896, 180)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,...,FACTOR6,FACTOR7,FACTOR8,FACTOR9,FACTOR10,FACTOR11,FACTOR12,FACTOR13,FACTOR14,FACTOR15,FACTOR16,FACTOR17,FACTOR18,y,t
0,-3.064007,-1.272259,-4.075844,1.492514,2.328039,-0.330240,-0.770389,0.035541,2.439325,-0.847266,-0.010803,-0.557613,0.203959,1.005977,-0.094114,...,V13,V3,V1,V1,V3,V2,V2,V5,V20,V9,V11,V1,V4,0,0
1,-4.574066,-3.541815,1.107371,0.447314,-0.471140,-0.567309,0.195963,0.383654,1.523472,-0.184596,-0.786622,0.294440,0.530339,1.146967,0.162051,...,V18,V2,V1,V1,V4,V2,V3,V6,V20,V9,V15,V4,V4,0,0
2,-1.751471,-2.039692,-3.788823,0.624226,0.564614,-0.713385,1.003502,1.582819,-1.461687,0.844975,0.570551,-0.259167,0.263823,1.274770,-0.065026,...,V20,V2,V1,V1,V4,V5,V2,V11,V20,V9,V1,V4,V4,0,1
3,-2.030089,-0.235720,-5.711960,1.071841,2.441541,0.758484,-0.534295,1.614289,-1.022909,-1.383025,-0.031077,1.052177,-0.072917,-2.071231,-0.675503,...,V13,V2,V1,V1,V3,V2,V2,V4,V20,V11,V1,V4,V4,0,1
4,-2.377857,-2.478670,-1.051946,2.812768,2.019267,0.095542,-0.033324,-1.332319,4.316526,-0.537967,-0.431669,-0.944672,0.267894,1.340286,-0.333540,...,V3,V2,V1,V1,V3,V2,V2,V7,V20,V2,V1,V2,V2,0,1


In [3]:
df.dtypes.value_counts()

float64    160
str         18
int64        2
Name: count, dtype: int64

In [4]:
pc_cols = [c for c in df.columns if c.startswith("PC")]
factor_cols = [c for c in df.columns if c.startswith("FACTOR")]
other = [c for c in df.columns if c not in pc_cols + factor_cols]
print(f"{len(pc_cols)} numeric PCA components: {pc_cols[0]} … {pc_cols[-1]}")
print(f"{len(factor_cols)} anonymised categorical factors: {factor_cols[0]} … {factor_cols[-1]}")
print("Other columns:", other)

160 numeric PCA components: PC1 … PC160
18 anonymised categorical factors: FACTOR1 … FACTOR18
Other columns: ['y', 't']


The columns are fully anonymised:

* **`PC1`–`PC160`** — the original numeric customer features (usage, tariff price, demographics…)
  were projected onto principal components by the data provider for privacy. Individual PCs have
  **no business meaning**.
* **`FACTOR1`–`FACTOR18`** — categorical features with generic level names (`V1`, `V2`, …).
* **`t`** and **`y`** — treatment and outcome (verified below).

## 2. Identify the causal variables

In [5]:
for col in [TREATMENT, OUTCOME]:
    print(col, df[col].value_counts().sort_index().to_dict())

t {0: 2886, 1: 9010}
y {0: 11488, 1: 408}


According to the dataset paper:

* **`t` (treatment)** — `1` if the customer was selected for a retention campaign: a call-centre
  contact offering a better-suited tariff/offer. `0` = held-out control group that was not contacted.
* **`y` (outcome)** — `1` if the customer **churned** in the two months following the campaign,
  `0` otherwise.

**Sign convention used throughout the project:** because `y = 1` is churn, a *successful*
retention intervention produces a **negative** effect on `y`. When targeting customers, scores are
expressed as the *reduction* in churn probability (positive = good), and this is stated wherever it is used.

## 3. Data quality

In [6]:
quality = pd.DataFrame({
    "missing values": [int(df.isna().sum().sum())],
    "duplicate rows": [int(df.duplicated().sum())],
    "duplicate rows (features only)": [int(df.drop(columns=[TREATMENT, OUTCOME]).duplicated().sum())],
})
quality

,missing values,duplicate rows,duplicate rows (features only)
0,0,0,0


In [7]:
desc = df[pc_cols].describe().T
desc[["mean", "std", "min", "max"]].iloc[[0, 1, 2, 9, 49, 99, 149, 155, 156, 157, 158, 159]]

,mean,std,min,max
PC1,-1.529076e-16,4.267859e+00,-5.855730e+00,2.206289e+02
PC2,1.529076e-16,3.592514e+00,-2.443104e+02,1.259816e+01
PC3,-9.843430e-16,3.050609e+00,-4.600993e+01,2.066914e+01
PC10,3.583773e-16,1.842393e+00,-6.996830e+00,1.397302e+01
PC50,-4.181069e-16,9.401930e-01,-3.568823e+01,6.043828e+00
PC100,2.389182e-17,5.023319e-01,-4.044695e+00,5.170008e+00
PC150,-3.624604e-17,1.368956e-02,-1.181202e+00,3.915475e-01
PC156,-1.603785e-16,2.961017e-03,-1.263309e-02,1.217833e-02
PC157,5.737537e-17,2.690819e-03,-1.427632e-01,7.960492e-02
PC158,-9.286662e-18,1.648614e-03,-7.543043e-02,6.147633e-02


PC means are ~0 by construction (PCA centres the data) and variances decline down the
component list, as expected. Some components have extreme maxima (e.g. `PC1` reaches ≈220 SDs),
which points to heavy-tailed original usage variables. Tree-based models are robust to this, and
logistic models use standardised inputs.

In [8]:
pc_sd = df[pc_cols].std()
print("PCs with ~zero variance:", list(pc_sd[pc_sd < 1e-8].index))

PCs with ~zero variance: ['PC160']


In [9]:
levels = pd.DataFrame({
    "n_levels": df[factor_cols].nunique(),
    "modal_level_share": df[factor_cols].apply(lambda s: s.value_counts(normalize=True).iloc[0]).round(4),
    "levels_with_<30_obs": df[factor_cols].apply(lambda s: int((s.value_counts() < 30).sum())),
})
levels

,n_levels,modal_level_share,levels_with_<30_obs
FACTOR1,18,0.4637,6
FACTOR2,12,0.6780,6
FACTOR3,1,1.0000,0
FACTOR4,20,0.2790,3
FACTOR5,4,0.5461,0
FACTOR6,20,0.2719,0
FACTOR7,3,0.7290,0
FACTOR8,2,0.9997,1
FACTOR9,4,0.6520,2
FACTOR10,4,0.5084,1


In [10]:
num, cat = feature_lists(df)
dropped = sorted(set(pc_cols + factor_cols) - set(num + cat), key=lambda c: (c[:2], int(''.join(filter(str.isdigit, c)))))
print(f"Usable covariates: {len(num)} numeric + {len(cat)} categorical")
print("Dropped as uninformative:", dropped)

Usable covariates: 158 numeric + 16 categorical
Dropped as uninformative: ['FACTOR3', 'FACTOR8', 'PC159', 'PC160']


**Data-quality findings**

* No missing values and no duplicated rows.
* `PC159`/`PC160` have numerically zero variance (the PCA had fewer informative dimensions than
  components), `FACTOR3` is constant and `FACTOR8` has only 4 customers outside its modal level.
  These are dropped from modelling by `feature_lists()`.
* Several factors have many sparse levels. Models pool levels with fewer than 50 customers
  (`make_preprocessor`), and the regression notebook pools levels with fewer than 100.

## 4. Outcome and treatment balance

In [11]:
summary = pd.DataFrame({
    "customers": df.groupby(TREATMENT).size(),
    "share": df[TREATMENT].value_counts(normalize=True).sort_index().round(4),
    "churners": df.groupby(TREATMENT)[OUTCOME].sum(),
    "churn_rate": df.groupby(TREATMENT)[OUTCOME].mean().round(4),
}).rename(index={0: "control (t=0)", 1: "treated (t=1)"})
summary.loc["total"] = [len(df), 1.0, df[OUTCOME].sum(), round(df[OUTCOME].mean(), 4)]
summary

,customers,share,churners,churn_rate
t,,,,
control (t=0),2886.0,0.2426,105.0,0.0364
treated (t=1),9010.0,0.7574,303.0,0.0336
total,11896.0,1.0000,408.0,0.0343


* **Outcome imbalance:** only **408 churners (3.4%)** in 11,896 customers. Every effect estimate
  rests on a small number of events, and only **105 churners are in the control group**. That number
  bounds how precisely anything about the counterfactual "no contact" world can be learned.
* **Treatment imbalance:** 75.7% treated vs 24.3% control. This was a deliberate design choice
  (most customers receive the campaign) and is not a problem in itself, but it makes the control
  arm small.
* The arm sizes and churn rates (3.64% control, 3.36% treated) match those reported in the
  dataset paper, which confirms the coding of `t` and `y`.

## 5. Treatment assignment — was it randomised?

The dataset paper describes a randomised control group withheld from each campaign. The data
however pools **several campaigns run between September and December 2020**, and no campaign
identifier is provided. If the treated/control ratio differed between campaigns, and campaign
populations differ, treatment would be randomised *within* campaigns but **not unconditionally
across the pooled data**. A quick check: does the treated share vary across customer segments?

In [12]:
for f in ["FACTOR4", "FACTOR18"]:
    g = df.groupby(f).agg(customers=(TREATMENT, "size"), treated_share=(TREATMENT, "mean"),
                          churn_rate=(OUTCOME, "mean"))
    display(g[g.customers >= 100].sort_values("treated_share").round(3))

,customers,treated_share,churn_rate
FACTOR4,,,
V16,468,0.632,0.045
V7,1278,0.667,0.031
V5,326,0.702,0.028
V1,175,0.709,0.046
V12,3319,0.714,0.037
V3,572,0.734,0.024
V8,978,0.779,0.037
V9,895,0.799,0.044
V11,868,0.836,0.041


,customers,treated_share,churn_rate
FACTOR18,,,
V6,526,0.643,0.042
V2,2300,0.689,0.029
V1,175,0.709,0.046
V4,3375,0.714,0.037
V7,5388,0.829,0.034


The treated share ranges from about **63% to 86%** across levels of `FACTOR4` and `FACTOR18`.
Under simple randomisation it would be ~76% in every segment, up to sampling noise (±3–4pp for
segments of this size). This is the first substantive finding: **the treatment was not assigned
with the same probability to every customer.** Notebooks 02 and 03 investigate it formally and
account for it.

## 6. Data dictionary

| Variable | Type | Role | Description |
|---|---|---|---|
| `t` | binary (int) | **Treatment T** | 1 = targeted by a retention campaign (call-centre contact with an offer); 0 = held-out control |
| `y` | binary (int) | **Outcome Y** | 1 = churned within 2 months after the campaign; 0 = retained |
| `PC1`–`PC158` | continuous | Covariates X | Principal components of the provider's numeric features (usage, billing, tariff, demographics); anonymised, no business meaning |
| `PC159`–`PC160` | continuous | — | Zero variance; dropped |
| `FACTOR1`–`FACTOR18` (excl. 3, 8) | categorical (3–20 levels) | Covariates X | Anonymised categorical features (e.g. region, tariff plan, contract type); level names are generic |
| `FACTOR3`, `FACTOR8` | categorical | — | Constant / near-constant; dropped |

* **Observations:** 11,896 customers, one row per customer-campaign.
* **Observation period:** campaigns September–December 2020; outcome measured over the 2 months after each campaign.
* **Missing values:** none.

## 7. Causal structure

```
  X (customer characteristics) ───────────────► Y (churn)
       │                                          ▲
       │  campaign / segment-dependent            │
       ▼  assignment probability                  │
  T (retention contact) ──────────────────────────┘
```

* **Target quantity:** the effect of contact on churn, i.e. the difference between each customer's
  churn if contacted, `Y(1)`, and if not contacted, `Y(0)`.
* **Why this is identifiable:** within campaigns, control customers were held out at random, so
  assignment depends only on the observed covariates `X` (via campaign and segment), not on the
  customer's unobserved propensity to churn. Under this assumption
  (**conditional ignorability**, `Y(0), Y(1) ⊥ T | X`), comparing treated and control customers
  *with similar X* identifies the causal effect.
* **What this means for the analysis:** the simple treated-vs-control difference is the natural
  first estimate. Because the assignment probability varies with `X`, it must be checked against
  covariate-adjusted estimates (regression adjustment, inverse-propensity weighting, doubly robust).
* **Key limitation:** the covariates are anonymised. The *existence* of heterogeneity can be
  established, but segments cannot be described in business language (e.g. "young customers on
  prepaid plans").